In [1]:
# Cargando el módulo Entrez de la biblioteca biopython y otras bibliotecas necesarias para esta sesión
from Bio import Entrez
import pandas as pd
from io import StringIO
import subprocess
import os

# Proporcionar correo electrónico
Entrez.mail= "leilanic@lcg.unam.mx"

In [24]:
# Buscar el UID del BioProject
project_acc = "PRJNA401856" # El código de BioProject q tiene guardado, lo normal es q solo se quite lo de letras y quede numérico
handle = Entrez.esearch(db="bioproject", term = project_acc) # 
search_results = Entrez.read(handle)

print(search_results)
handle.close() # Cerrando la petición, es elemental siempre hacerlo

# Crtl + Enter para correr el cuadrito de codigo (hay q estar paradas en la celda pa q funcione)

# El "IdList" es el resultado q nos interesa (q básicamente solo es el código de BioProject que habíamos metido sin letras)
# El resultado impreso en pantalla luce como un diccionario de chingos de cosas

{'Count': '1', 'RetMax': '1', 'RetStart': '0', 'IdList': ['401856'], 'TranslationSet': [], 'TranslationStack': [{'Term': 'PRJNA401856[All Fields]', 'Field': 'All Fields', 'Count': '1', 'Explode': 'N'}, 'GROUP'], 'QueryTranslation': 'PRJNA401856[All Fields]'}


In [25]:
# Extraer solo el UID
project_uid = search_results["IdList"][0] # Como el "IdList" de los resultados es una lista, por eso especificamos que queremos el primer/
#resultados con el [0]

print(project_uid) # Imprimimos el resultado sacado

401856


In [26]:
# Obtener resumen del BioProject

handle = Entrez.esummary(db= "bioproject", id = project_uid, retmode = "txt") # Esto solo es la consulta, para recuperarlo es otra cosa
project_record = Entrez.read(handle) # Esto es para recuperarlo
print(project_record)

handle.close() # Es nomas por buena practica pa q no se atonte la compu

{'DocumentSummarySet': DictElement({'DocumentSummary': [DictElement({'TaxId': '223283', 'Project_Id': '401856', 'Project_Acc': 'PRJNA401856', 'Project_Type': 'Primary submission', 'Project_Data_Type': 'Transcriptome or Gene expression', 'Sort_By_ProjectType': '856426', 'Sort_By_DataType': '847437', 'Sort_By_Organism': '609150', 'Project_Subtype': '', 'Project_Target_Scope': 'Multiisolate', 'Project_Target_Material': 'Transcriptome', 'Project_Target_Capture': 'Whole', 'Project_MethodType': 'Sequencing', 'Project_Method': '', 'Project_Objectives_List': [{'Project_ObjectivesType': 'Expression', 'Project_Objectives': ''}], 'Registration_Date': '2017/09/05 00:00', 'Project_Name': 'Pseudomonas syringae pv. tomato str. DC3000', 'Project_Title': 'In planta bacterial RNA-Seq with bacterial isolation.', 'Project_Description': 'Plant pathogens can cause serious diseases that impact global agriculture1. Understanding how the plant immune system naturally restricts pathogen infection holds a key to

In [27]:
# Seleccionando la información relacionada al proyecto
proj = project_record["DocumentSummarySet"]["DocumentSummary"][0] # Siempre es DocumentSummarySet>DocumentSummary>Numeroderesultado


# Mostrando todos los campos del diccionario
print(proj.keys())
print("\n=== Información del BioProject en print ===")
for key, value in proj.items(): # Con esto obtenemos lo de absolutamente todos los campos
    print(key, "===", value)



# Accediendo a algunos campos
print("\n=== Información del BioProject ===")
print("Organismo:", proj['Organism_Name'])
print("Supergrupo:", proj['Supergroup'])


# Chale q no llenen todos los campos

dict_keys(['TaxId', 'Project_Id', 'Project_Acc', 'Project_Type', 'Project_Data_Type', 'Sort_By_ProjectType', 'Sort_By_DataType', 'Sort_By_Organism', 'Project_Subtype', 'Project_Target_Scope', 'Project_Target_Material', 'Project_Target_Capture', 'Project_MethodType', 'Project_Method', 'Project_Objectives_List', 'Registration_Date', 'Project_Name', 'Project_Title', 'Project_Description', 'Keyword', 'Relevance_Agricultural', 'Relevance_Medical', 'Relevance_Industrial', 'Relevance_Environmental', 'Relevance_Evolution', 'Relevance_Model', 'Relevance_Other', 'Organism_Name', 'Organism_Strain', 'Organism_Label', 'Sequencing_Status', 'Submitter_Organization', 'Submitter_Organization_List', 'Supergroup'])

=== Información del BioProject en print ===
TaxId === 223283
Project_Id === 401856
Project_Acc === PRJNA401856
Project_Type === Primary submission
Project_Data_Type === Transcriptome or Gene expression
Sort_By_ProjectType === 856426
Sort_By_DataType === 847437
Sort_By_Organism === 609150
Proj

## SRA directamente del BioProjet, no de GEO

In [11]:
# Obtener los registros SRA relacionados con el BioProject 
# pARA CUANDO EL BioProject no tiene un GEO
handle = Entrez.elink(dbfrom = "bioproject", db = "sra", id = project_uid) # El archivo q yo quiero viene de BioProjects, y quiero consultar su SRA
linksSRA= Entrez.read(handle)
#print(linksSRA)


# Seleccionar solo los IDs
sra_uids = [link["Id"] for link in linksSRA[0]['LinkSetDb'][0]['Link']] 
print(sra_uids[:5]) # Imprime los primeros 5 identificadores encontrados
print(len(sra_uids)) # Reporta la cantidad de identificadores de experimentos totales q hay 

NameError: name 'project_uid' is not defined

In [29]:
# Obtener información de los registros SRA
ids = sra_uids

# Haciendo la consulta
handle = Entrez.efetch(db = "sra", id= ids, rettype = "runinfo", retmode= "text") # La base de datos consultada(db).  . Decir cual es el tipo de informacion requerida(tuninfo). En q modo la queremos(txt)

# Leyendo la consulta
runinfo = handle.read().decode("utf-8")
#print(runinfo)
handle.close()

# Creando un dataframe de los Run
df_runinfo = pd.read_csv(StringIO(runinfo)) # Para q se vuelva dataframe
print(df_runinfo.shape) 

(114, 47)
Index(['Run', 'ReleaseDate', 'LoadDate', 'spots', 'bases', 'spots_with_mates',
       'avgLength', 'size_MB', 'AssemblyName', 'download_path', 'Experiment',
       'LibraryName', 'LibraryStrategy', 'LibrarySelection', 'LibrarySource',
       'LibraryLayout', 'InsertSize', 'InsertDev', 'Platform', 'Model',
       'SRAStudy', 'BioProject', 'Study_Pubmed_id', 'ProjectID', 'Sample',
       'BioSample', 'SampleType', 'TaxID', 'ScientificName', 'SampleName',
       'g1k_pop_code', 'source', 'g1k_analysis_group', 'Subject_ID', 'Sex',
       'Disease', 'Tumor', 'Affection_Status', 'Analyte_Type',
       'Histological_Type', 'Body_Site', 'CenterName', 'Submission',
       'dbgap_study_accession', 'Consent', 'RunHash', 'ReadHash'],
      dtype='str')


In [30]:
# Imprimiendo solo algunos campos del dataframe
print( df_runinfo[ ["Run", "Experiment", "Sample", "BioSample", "BioProject"] ] ) # Le quite el head, pq queremos ver todas las samples

            Run  Experiment      Sample     BioSample   BioProject
0    SRR6003772  SRX3159054  SRS2490379  SAMN07604127  PRJNA401856
1    SRR6003773  SRX3159055  SRS2490378  SAMN07604126  PRJNA401856
2    SRR6003774  SRX3159056  SRS2490380  SAMN07604125  PRJNA401856
3    SRR6003775  SRX3159057  SRS2490381  SAMN07604124  PRJNA401856
4    SRR6003776  SRX3159058  SRS2490382  SAMN07604123  PRJNA401856
..          ...         ...         ...           ...          ...
109  SRR6003767  SRX3159049  SRS2490373  SAMN07604227  PRJNA401856
110  SRR6003768  SRX3159050  SRS2490375  SAMN07604131  PRJNA401856
111  SRR6003769  SRX3159051  SRS2490374  SAMN07604130  PRJNA401856
112  SRR6003770  SRX3159052  SRS2490377  SAMN07604129  PRJNA401856
113  SRR6003771  SRX3159053  SRS2490376  SAMN07604128  PRJNA401856

[114 rows x 5 columns]


In [31]:
# Pasando solo los ids a una lista
srr_ids = df_runinfo["Run"].tolist() 
print("Número de SRR:", len(srr_ids)) 
print(srr_ids[:10])

Número de SRR: 114
['SRR6003772', 'SRR6003773', 'SRR6003774', 'SRR6003775', 'SRR6003776', 'SRR6003777', 'SRR6003778', 'SRR6003779', 'SRR6003780', 'SRR6003781']


In [32]:
# Asociando los Runs a los Experimentos
srr_por_srx = df_runinfo.groupby("Experiment")["Run"].apply(list)
print(srr_por_srx.head(20))

Experiment
SRX3159000    [SRR6003744]
SRX3159001    [SRR6003745]
SRX3159002    [SRR6003718]
SRX3159003    [SRR6003719]
SRX3159004    [SRR6003720]
SRX3159005    [SRR6003721]
SRX3159006    [SRR6003722]
SRX3159007    [SRR6003723]
SRX3159008    [SRR6003724]
SRX3159009    [SRR6003725]
SRX3159010    [SRR6003726]
SRX3159011    [SRR6003727]
SRX3159012    [SRR6003728]
SRX3159013    [SRR6003729]
SRX3159014    [SRR6003730]
SRX3159015    [SRR6003731]
SRX3159016    [SRR6003732]
SRX3159017    [SRR6003733]
SRX3159018    [SRR6003734]
SRX3159019    [SRR6003735]
Name: Run, dtype: object


In [12]:
# Revisar enlaces desde BioProject hacia GEO (GDS), con el uid del bioproject
handle = Entrez.elink(dbfrom = "bioproject", db = "sra", id = project_uid) 
linksBioProj = Entrez.read(handle)




print(linksBioProj)
geo_ids = [link["Id"] for link in linksBioProj[0]["LinkSetDb"][0]["Link"]] # A este campo, a este Link, a este Id
print("UIDs GEO asociados:", geo_ids[:5], "... total:", len(geo_ids))   # Efectivamente tuvimos asociados los 156 que decía en la página

NameError: name 'project_uid' is not defined